In [1]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = "../data/UK_Accident.csv"
df = pd.read_csv(file_path)

# Remove unnecessary columns
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

if "Accident_Index" in df.columns:
    df.drop(columns=["Accident_Index"], inplace=True)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (111, 31)


,Location_Easting_OSGR,Location_Northing_OSGR,Longitude,Latitude,Police_Force,Accident_Severity,Number_of_Vehicles,Number_of_Casualties,Date,Day_of_Week,...,Pedestrian_Crossing-Physical_Facilities,Light_Conditions,Weather_Conditions,Road_Surface_Conditions,Special_Conditions_at_Site,Carriageway_Hazards,Urban_or_Rural_Area,Did_Police_Officer_Attend_Scene_of_Accident,LSOA_of_Accident_Location,Year
0,525680.0,178240.0,-0.191170,51.489096,1,2,1,1,04/01/2005,3,...,Zebra crossing,Daylight: Street light present,Raining without high winds,Wet/Damp,NaN,NaN,1,Yes,E01002849,2005
1,524170.0,181650.0,-0.211708,51.520075,1,3,1,1,05/01/2005,4,...,Pedestrian phase at traffic signal junction,Darkness: Street lights present and lit,Fine without high winds,Dry,NaN,NaN,1,Yes,E01002909,2005
2,524520.0,182240.0,-0.206458,51.525301,1,3,2,1,06/01/2005,5,...,No physical crossing within 50 meters,Darkness: Street lights present and lit,Fine without high winds,Dry,NaN,NaN,1,Yes,E01002857,2005
3,526900.0,177530.0,-0.173862,51.482442,1,3,1,1,07/01/2005,6,...,No physical crossing within 50 meters,Daylight: Street light present,Fine without high winds,Dry,NaN,NaN,1,Yes,E01002840,2005
4,528060.0,179040.0,-0.156618,51.495752,1,3,1,1,10/01/2005,2,...,No physical crossing within 50 meters,Darkness: Street lighting unknown,Fine without high winds,Wet/Damp,NaN,NaN,1,Yes,E01002863,2005


In [3]:
# Target variable
target = "Accident_Severity"

# Features for the first ML model
features = [
    "Number_of_Vehicles",
    "Number_of_Casualties",
    "Police_Force",
    "Speed_limit",
    "Light_Conditions",
    "Weather_Conditions",
    "Road_Surface_Conditions",
    "Special_Conditions_at_Site",
    "Carriageway_Hazards",
    "Urban_or_Rural_Area",
    "Did_Police_Officer_Attend_Scene_of_Accident",
    "Year"
]

# Make sure all selected columns exist
missing_features = [
    feature for feature in features
    if feature not in df.columns
]

if missing_features:
    print("Missing features:", missing_features)
    raise ValueError("One or more selected columns are not available.")

# Create feature and target datasets
X = df[features].copy()
y = df[target].copy()

print("Number of features:", len(features))
print("\nFeatures:")
for feature in features:
    print("-", feature)

print("\nTarget:", target)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Number of features: 12

Features:
- Number_of_Vehicles
- Number_of_Casualties
- Police_Force
- Speed_limit
- Light_Conditions
- Weather_Conditions
- Road_Surface_Conditions
- Special_Conditions_at_Site
- Carriageway_Hazards
- Urban_or_Rural_Area
- Did_Police_Officer_Attend_Scene_of_Accident
- Year

Target: Accident_Severity

X shape: (111, 12)
y shape: (111,)


In [4]:
print("=" * 60)
print("FEATURE DATA TYPES")
print("=" * 60)

print(X.dtypes)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_values = X.isnull().sum()

if missing_values.sum() == 0:
    print("No missing values found.")
else:
    print(missing_values[missing_values > 0].sort_values(ascending=False))

# Separate numerical and categorical features
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\n" + "=" * 60)
print("NUMERICAL FEATURES")
print("=" * 60)

print(numerical_features)

print("\n" + "=" * 60)
print("CATEGORICAL FEATURES")
print("=" * 60)

print(categorical_features)

FEATURE DATA TYPES
Number_of_Vehicles                             int64
Number_of_Casualties                           int64
Police_Force                                   int64
Speed_limit                                    int64
Light_Conditions                                 str
Weather_Conditions                               str
Road_Surface_Conditions                          str
Special_Conditions_at_Site                       str
Carriageway_Hazards                              str
Urban_or_Rural_Area                            int64
Did_Police_Officer_Attend_Scene_of_Accident      str
Year                                           int64
dtype: object

MISSING VALUES
Carriageway_Hazards           110
Special_Conditions_at_Site    108
dtype: int64

NUMERICAL FEATURES
['Number_of_Vehicles', 'Number_of_Casualties', 'Police_Force', 'Speed_limit', 'Urban_or_Rural_Area', 'Year']

CATEGORICAL FEATURES
['Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Special_Cond

C:\Users\HP\AppData\Local\Temp\ipykernel_23236\194626177.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical data processing
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical data processing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully! ✅")

Preprocessing pipeline created successfully! ✅


In [6]:
from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset split completed!")
print("\nTraining samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())


# Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the already-fitted preprocessing to test data
X_test_processed = preprocessor.transform(X_test)

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

Dataset split completed!

Training samples: 88
Testing samples : 23

Training target distribution:
Accident_Severity
2     7
3    81
Name: count, dtype: int64

Testing target distribution:
Accident_Severity
2     2
3    21
Name: count, dtype: int64

PREPROCESSING COMPLETED
Processed training shape: (88, 24)
Processed testing shape : (23, 24)


In [7]:
print("=" * 60)
print("FINAL PREPROCESSED DATA")
print("=" * 60)

print("Training data shape:", X_train_processed.shape)
print("Testing data shape :", X_test_processed.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape :", y_test.shape)

print("\nChecking for missing values after preprocessing:")

print(
    "Training missing values:",
    np.isnan(X_train_processed).sum()
)

print(
    "Testing missing values:",
    np.isnan(X_test_processed).sum()
)

# Convert processed data to DataFrame for inspection
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("\nProcessed feature count:", len(feature_names))

print("\nFirst 5 processed training records:")
display(X_train_processed_df.head())

print("\n" + "=" * 60)
print("FEATURE ENGINEERING & PREPROCESSING COMPLETED ✅")
print("=" * 60)

FINAL PREPROCESSED DATA
Training data shape: (88, 24)
Testing data shape : (23, 24)

Training target shape: (88,)
Testing target shape : (23,)

Checking for missing values after preprocessing:
Training missing values: 0
Testing missing values: 0

Processed feature count: 24

First 5 processed training records:


,numerical__Number_of_Vehicles,numerical__Number_of_Casualties,numerical__Police_Force,numerical__Speed_limit,numerical__Urban_or_Rural_Area,numerical__Year,categorical__Light_Conditions_Darkness: Street lighting unknown,categorical__Light_Conditions_Darkness: Street lights present and lit,categorical__Light_Conditions_Daylight: Street light present,categorical__Weather_Conditions_Fine without high winds,...,categorical__Road_Surface_Conditions_Dry,categorical__Road_Surface_Conditions_Frost/Ice,categorical__Road_Surface_Conditions_Snow,categorical__Road_Surface_Conditions_Wet/Damp,categorical__Special_Conditions_at_Site_Auto traffic signal partly defective,categorical__Special_Conditions_at_Site_Ol or diesel,categorical__Special_Conditions_at_Site_Roadworks,categorical__Carriageway_Hazards_Other object in carriageway,categorical__Did_Police_Officer_Attend_Scene_of_Accident_No,categorical__Did_Police_Officer_Attend_Scene_of_Accident_Yes
6,0.407307,-0.337526,0.0,0.0,0.0,0.107211,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
39,0.407307,-0.337526,0.0,0.0,0.0,0.107211,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
59,-1.086151,-0.337526,0.0,0.0,0.0,0.107211,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
72,1.900764,-0.337526,0.0,0.0,0.0,0.107211,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
50,0.407307,-0.337526,0.0,0.0,0.0,0.107211,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0



FEATURE ENGINEERING & PREPROCESSING COMPLETED ✅
